# Lesson 3.4 — ADQL Analytics Tables

## Learning Objectives
- Write queries against `transactions`, `browser_records`, and `application_logs` tables
- Understand the schema and key fields of each table
- Use table-specific ADQL patterns for different monitoring scenarios

**Reference**: [ADQL Reference](https://help.splunk.com/en/appdynamics-saas/analytics/26.2.0/analytics/adql-reference/adql-queries) | [Grafana Analytics Query Examples](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#analytics-query-examples)

## 1. The `transactions` Table

This is the most commonly used ADQL table. It contains one record per Business Transaction execution.

### Key Fields

| Field | Type | Description |
|-------|------|-------------|
| `eventTimestamp` | Long (ms) | When the transaction started |
| `transactionName` | String | BT name (e.g., `/checkout`) |
| `applicationName` | String | AppDynamics application name |
| `tierName` | String | Tier where the BT was detected |
| `nodeName` | String | Specific node that processed the request |
| `responseTime` | Long (ms) | Total response time |
| `userExperience` | String | NORMAL, SLOW, VERY_SLOW, STALL, ERROR |
| `errorOccurred` | Boolean | Whether an error occurred |
| `errorMessage` | String | Error message (if any) |
| `httpStatus` | Integer | HTTP response code |
| `httpRequestMethod` | String | GET, POST, PUT, DELETE |
| `requestGUID` | String | Unique correlation ID |

Additional fields (if Business Transaction Analytics is configured):
- Custom data fields you've instrumented via the SDK
- User identifiers, product IDs, cart values, etc.

In [ ]:
# Practical ADQL queries for the transactions table

transactions_queries = {
    "Top slowest BTs": """
SELECT transactionName, max(responseTime) AS maxResponseTime
FROM transactions
ORDER BY maxResponseTime DESC
LIMIT 10
""",
    "Error diagnosis": """
SELECT transactionName, responseTime, errorMessage, nodeName
FROM transactions
WHERE errorOccurred = true
ORDER BY eventTimestamp DESC
LIMIT 50
""",
    "Volume by BT": """
SELECT transactionName, count(*) AS total, avg(responseTime) AS avgResponseTime
FROM transactions
ORDER BY total DESC
""",
    "Very slow transactions with user data": """
SELECT transactionName, responseTime, nodeName
FROM transactions
WHERE userExperience = 'VERY_SLOW'
ORDER BY responseTime DESC
LIMIT 25
""",
    "HTTP error analysis": """
SELECT transactionName, httpStatus, count(*) AS cnt
FROM transactions
WHERE httpStatus >= 400
ORDER BY cnt DESC
""",
    "Time-series error rate (for chart)": """
SELECT series(eventTimestamp, '1m'), count(*) AS errors
FROM transactions
WHERE errorOccurred = true
""",
}

print("=== Practical transactions table queries ===")
for name, query in transactions_queries.items():
    print(f"\n-- {name}")
    print(query.strip())

## 2. The `browser_records` Table

Contains one record per browser page load or AJAX call collected by the Browser RUM agent (`adrum.js`).

### Key Fields

| Field | Type | Description |
|-------|------|-------------|
| `eventTimestamp` | Long (ms) | When the page load completed |
| `pageUrl` | String | Full URL of the page |
| `domReadyTime` | Long (ms) | DOM content loaded time |
| `pageLoadTime` | Long (ms) | Total page load time |
| `responseTime` | Long (ms) | Time to First Byte (TTFB) |
| `frontEndResponseTime` | Long (ms) | Client-side processing time |
| `serverSideResponseTime` | Long (ms) | Correlated server-side BT time |
| `networkTime` | Long (ms) | Network time |
| `country` | String | User's country |
| `city` | String | User's city |
| `browser` | String | Browser name |
| `userExperience` | String | NORMAL, SLOW, VERY_SLOW, ERROR |
| `errorOccurred` | Boolean | JavaScript error occurred |

In [ ]:
# Practical ADQL queries for the browser_records table
# (from Grafana AppDynamics Datasource official docs)

browser_queries = {
    "Slowest pages by avg DOM ready time": """
SELECT pageUrl, avg(domReadyTime) AS avgDomReady, count(*) AS total
FROM browser_records
ORDER BY avgDomReady DESC
""",
    "Page performance by country": """
SELECT country, avg(pageLoadTime) AS avgLoadTime, count(*) AS sessions
FROM browser_records
ORDER BY avgLoadTime DESC
""",
    "Browser error analysis": """
SELECT pageUrl, browser, count(*) AS errors
FROM browser_records
WHERE errorOccurred = true
ORDER BY errors DESC
""",
    "Page load time series (for chart)": """
SELECT series(eventTimestamp, '5m'), avg(pageLoadTime) AS avgLoad
FROM browser_records
""",
    "User experience distribution": """
SELECT userExperience, count(*) AS cnt
FROM browser_records
ORDER BY cnt DESC
""",
    "Mobile vs Desktop performance": """
SELECT deviceType, avg(pageLoadTime) AS avgLoadTime, count(*) AS sessions
FROM browser_records
ORDER BY sessions DESC
"""
}

print("=== Practical browser_records table queries ===")
for name, query in browser_queries.items():
    print(f"\n-- {name}")
    print(query.strip())

## 3. The `application_logs` Table

Contains application log events collected by the Analytics Agent from your log files.

### Key Fields

| Field | Type | Description |
|-------|------|-------------|
| `eventTimestamp` | Long (ms) | Log event time |
| `MSGTXT` | String | The log message text |
| `MSGTY` | String | Log level: E (Error), W (Warning), I (Info), D (Debug) |
| `Application` | String | Application name |
| `serverName` | String | Server/host where the log was generated |
| `requestGUID` | String | Correlation ID linking log to a transaction |

> **Note**: Exact field names may vary depending on your log format and Analytics Agent configuration.

In [ ]:
# Practical ADQL queries for application_logs table
# (from Grafana AppDynamics Datasource official docs)

log_queries = {
    "Recent error logs": """
SELECT eventTimestamp, MSGTY, MSGTXT, Application
FROM application_logs
WHERE MSGTY = 'E'
ORDER BY eventTimestamp DESC
LIMIT 50
""",
    "Log error count by server": """
SELECT serverName, count(*) AS errorCount
FROM application_logs
WHERE MSGTY = 'E'
ORDER BY errorCount DESC
""",
    "Error log timeline": """
SELECT series(eventTimestamp, '1m'), count(*) AS errors
FROM application_logs
WHERE MSGTY = 'E'
""",
    "Logs containing specific keyword": """
SELECT eventTimestamp, serverName, MSGTXT
FROM application_logs
WHERE MSGTXT CONTAINS 'OutOfMemoryError'
ORDER BY eventTimestamp DESC
LIMIT 20
""",
}

print("=== Practical application_logs table queries ===")
for name, query in log_queries.items():
    print(f"\n-- {name}")
    print(query.strip())

## 4. Choosing the Right Table

Use this decision guide:

```
What are you investigating?
│
├── Server-side BT performance, errors, user experience?
│   └── USE: transactions
│
├── Browser / client-side page load, AJAX, JS errors?
│   └── USE: browser_records
│
├── Mobile app crashes, network calls?
│   └── USE: mobile_snapshots
│
├── Application log messages, stack traces?
│   └── USE: application_logs
│
└── Custom business events (e.g., order value, cart items)?
    └── USE: custom_events (or enriched transaction fields)
```

In [ ]:
# EXERCISE 3.4: Cross-table analysis scenario
#
# Scenario: A user reports that the checkout page is slow on their iPhone.
# You need to investigate using Analytics.
#
# TASK: For each step below, write the correct ADQL query.

investigation_steps = [
    {
        "step": 1,
        "question": "How many browser sessions came from iOS devices in the last hour?",
        "table": "browser_records",
        "adql": """-- TODO: Fill in the ADQL query
SELECT ??? FROM browser_records WHERE ???"""
    },
    {
        "step": 2,
        "question": "What is the average page load time for iOS vs Android vs Desktop?",
        "table": "browser_records",
        "adql": """-- TODO: Fill in the ADQL query
SELECT ??? FROM browser_records ???"""
    },
    {
        "step": 3,
        "question": "Find server-side /checkout transactions where responseTime > 3000ms",
        "table": "transactions",
        "adql": """-- TODO: Fill in the ADQL query
SELECT ??? FROM transactions WHERE ???"""
    },
    {
        "step": 4,
        "question": "Show error log messages containing 'payment' from the last 30 mins",
        "table": "application_logs",
        "adql": """-- TODO: Fill in the ADQL query
SELECT ??? FROM application_logs WHERE ???"""
    },
]

for step in investigation_steps:
    print(f"Step {step['step']}: {step['question']}")
    print(f"  Table: {step['table']}")
    print(f"  Query:")
    for line in step['adql'].split('\n'):
        print(f"    {line}")
    print()

## Summary

- **`transactions`**: Server-side BT performance, errors, user experience — most-used table
- **`browser_records`**: Client-side page performance, AJAX calls, browser/device/geo data
- **`application_logs`**: Log messages correlated to transactions via `requestGUID`
- **`mobile_snapshots`**: Mobile APM data — crashes, network requests, screen performance
- Choose the table based on *where the symptom appears*: server, browser, mobile, or logs

---

**Next**: [Lesson 3.5 — Variables in ADQL Queries](lesson-3.5-adql-variables.ipynb)